# Regression checks for main or any tag

Run this notebook inside the repository using Python 3.11+ with Git available.
Select a **local** branch, tag, or commit below. `main` means local main;
`origin/main` means the last fetched remote main. Fetch updates separately when wanted.

The runner clones the exact commit into a temporary directory, installs that source and its
dev dependencies in a fresh virtual environment, verifies the import location, performs an
MCP tool-list/cache round trip, and runs the selected commit's full pytest suite by default.
It never switches your working branch, includes uncommitted changes, pushes, or publishes.
Dependency installation requires package-index access; dependency versions are not locked.

Use `FOCUSED = True` for just compatibility, Greek normalization, recorded CTS fixtures,
and packaging tests. Optional live retrieval is separate from offline tests. Older refs missing
these tests or the modern MCP contract fail explicitly rather than silently passing.
All temporary files are removed after the run, including on failure. Output remains below.


In [ ]:
REF = "v1_1_0"  # Also accepts "main", "v1_1_0", "origin/main", or a commit SHA.
FOCUSED = False
LIVE = False

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "tests/run_ref_regression.py").is_file()
)
command = [sys.executable, str(REPO / "tests/run_ref_regression.py"), "--ref", REF]
if FOCUSED:
    command.append("--focused")
if LIVE:
    command.append("--live")
# Stream progress into the notebook; any failing check makes this cell fail.
with subprocess.Popen(
    command,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"},
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
assert return_code == 0, (
    f"Regression run failed (exit {return_code}); see output above."
)